# Notebook 15: Quantization and Model Compression

**Learning Objectives:**
- Understand numeric precision (FP32, FP16, INT8) and its impact on model size
- Apply PyTorch dynamic quantization to reduce model size and latency
- Export models to ONNX format for optimized inference
- Benchmark quantized vs. original models on accuracy, speed, and size
- Know when to use each compression technique

**Prerequisites:** Notebook 11 (Performance and Caching)

## Prerequisites

### Hardware Requirements

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum, 8GB recommended |
| **Storage** | 3GB free (for model copies) |
| **GPU** | Not required (all techniques work on CPU) |

### Software Requirements
- Python 3.8+
- Libraries: `transformers`, `torch`, `onnx`, `onnxruntime`, `optimum`
- See `requirements.txt` for full list

### Optional Installation
```bash
pip install onnx onnxruntime optimum[onnxruntime]
```

## Expected Behaviors

### Dynamic Quantization (PyTorch)
- Model size reduces by ~2-4x (FP32 to INT8)
- Inference speed improves 1.5-3x on CPU
- Accuracy loss is typically <1%
- Works on any CPU (no special hardware needed)

### ONNX Export
- First export takes 10-30 seconds
- ONNX file is roughly the same size as the PyTorch model
- ONNX Runtime inference is typically 1.5-2x faster than PyTorch

### Benchmarking
- Results vary by hardware — your numbers will differ from examples
- GPU users may see smaller speedups from quantization (GPUs are already fast)
- Memory savings are consistent across hardware

### Common Issues
- **ONNX export fails**: Some model architectures are not fully supported
- **onnxruntime not found**: Install with `pip install onnxruntime`
- **Accuracy drop**: Expected for aggressive quantization; measure before deploying

## Overview

**Model compression** reduces model size and inference cost while preserving accuracy. This is critical for:
- **Edge deployment**: Running models on phones, IoT devices, or laptops
- **Cost reduction**: Smaller models use less compute and memory
- **Latency**: Faster inference means better user experience

**Techniques covered in this notebook:**

| Technique | How It Works | Size Reduction | Speed Improvement |
|-----------|-------------|----------------|-------------------|
| **FP16 (Half Precision)** | Reduce float precision from 32 to 16 bits | 2x | 1-2x (GPU only) |
| **Dynamic Quantization** | Quantize weights to INT8 at runtime | 2-4x | 1.5-3x (CPU) |
| **ONNX Export** | Convert to optimized runtime format | ~1x | 1.5-2x |
| **ONNX + Quantization** | Combine ONNX with INT8 quantization | 2-4x | 2-4x |

**What we won't cover** (requires GPU-specific setup):
- Static quantization (requires calibration dataset)
- bitsandbytes 4-bit/8-bit (Linux + CUDA only)
- GPTQ/AWQ (requires specific GPU libraries)

## Setup and Installation

In [ ]:
# Import libraries
import os
import random
import sys
import time
import numpy as np
import pandas as pd
import torch
import torch.quantization
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
)
import warnings
warnings.filterwarnings('ignore')

# Check for optional dependencies
try:
    import onnx
    import onnxruntime as ort
    ONNX_AVAILABLE = True
except ImportError:
    ONNX_AVAILABLE = False
    print("ONNX not available. Install with: pip install onnx onnxruntime")

try:
    from optimum.onnxruntime import ORTModelForSequenceClassification
    OPTIMUM_AVAILABLE = True
except ImportError:
    OPTIMUM_AVAILABLE = False
    print("Optimum not available. Install with: pip install optimum[onnxruntime]")

# Set seed for reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if ONNX_AVAILABLE:
    print(f"ONNX Runtime: {ort.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Understanding Numeric Precision

Neural network weights are stored as floating-point numbers. The precision (number of bits per weight) directly determines model size and compute requirements. Reducing precision is the core idea behind quantization — we trade a small amount of numerical accuracy for major savings in size and speed.

In [ ]:
# Demonstrate precision differences
print("=== NUMERIC PRECISION ===")
print(f"{'Type':<12} {'Bits':<6} {'Range':<35} {'Example'}")
print("-" * 75)

# FP32
fp32_val = torch.tensor(3.14159265, dtype=torch.float32)
print(f"{'FP32':<12} {'32':<6} {'+-3.4e38':<35} {fp32_val.item():.8f}")

# FP16
fp16_val = fp32_val.half()
print(f"{'FP16':<12} {'16':<6} {'+-65504':<35} {fp16_val.item():.8f}")

# BF16
bf16_val = fp32_val.bfloat16()
print(f"{'BF16':<12} {'16':<6} {'+-3.4e38 (less precise)':<35} {bf16_val.item():.8f}")

# INT8
print(f"{'INT8':<12} {'8':<6} {'-128 to 127':<35} {'3 (rounded)'}")

# Show size impact
print("\n=== SIZE IMPACT (for 100M parameter model) ===")
params = 100_000_000
size_df = pd.DataFrame({
    "Precision": ["FP32", "FP16/BF16", "INT8", "INT4"],
    "Bits": [32, 16, 8, 4],
    "Size (MB)": [params * 4 / 1e6, params * 2 / 1e6, params * 1 / 1e6, params * 0.5 / 1e6],
    "Reduction": ["1x (baseline)", "2x", "4x", "8x"],
})
print(size_df.to_string(index=False))

## Part 2: Loading the Base Model

We'll use DistilBERT for sentiment analysis as our base model throughout this notebook. It's small enough to experiment with quickly but large enough that compression techniques make a meaningful difference.

In [ ]:
# Load base model and tokenizer
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()  # Set to evaluation mode

# Model size utility
def get_model_size_mb(model: torch.nn.Module) -> float:
    """Calculate model size in MB by summing parameter memory.

    Args:
        model: A PyTorch model.

    Returns:
        Model size in megabytes.
    """
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)


original_size = get_model_size_mb(model)
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Size: {original_size:.1f} MB")

In [ ]:
# Baseline inference utility
def predict_sentiment(
    text: str,
    model: torch.nn.Module,
    tokenizer: AutoTokenizer,
) -> dict[str, float]:
    """Run sentiment prediction with a PyTorch model.

    Args:
        text: Input text to classify.
        model: The classification model.
        tokenizer: The tokenizer for the model.

    Returns:
        Dictionary with label and confidence score.
    """
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    label_id = probs.argmax().item()
    labels = ["NEGATIVE", "POSITIVE"]
    return {"label": labels[label_id], "confidence": probs[0][label_id].item()}


# Test baseline
test_texts = [
    "This movie was absolutely fantastic!",
    "Terrible waste of time.",
    "It was okay, nothing special.",
]

print("=== BASELINE PREDICTIONS ===")
for text in test_texts:
    result = predict_sentiment(text, model, tokenizer)
    print(f"  '{text}'")
    print(f"    -> {result['label']} ({result['confidence']:.4f})")

## Part 3: Half Precision (FP16)

The simplest compression technique is converting FP32 weights to FP16. This halves the model size with virtually no accuracy loss for inference. On GPUs, FP16 also speeds up computation. On CPU, the size benefit remains but speed improvement is minimal.

In [ ]:
# Convert to FP16
model_fp16 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model_fp16.eval()

fp16_size = get_model_size_mb(model_fp16)
print(f"=== FP16 MODEL ===")
print(f"Original size: {original_size:.1f} MB (FP32)")
print(f"FP16 size:     {fp16_size:.1f} MB")
print(f"Reduction:     {original_size / fp16_size:.1f}x")

# Verify predictions match
print("\n=== FP16 PREDICTIONS ===")
for text in test_texts:
    # FP16 models need float16 inputs on CPU
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model_fp16(**inputs)
    probs = torch.softmax(outputs.logits.float(), dim=-1)  # Cast back to FP32 for softmax
    label_id = probs.argmax().item()
    labels = ["NEGATIVE", "POSITIVE"]
    print(f"  '{text}' -> {labels[label_id]} ({probs[0][label_id].item():.4f})")

del model_fp16  # Free memory

## Part 4: Dynamic Quantization (PyTorch)

Dynamic quantization converts model weights from FP32 to INT8 and performs computations in INT8 during inference. "Dynamic" means activations are quantized on-the-fly (no calibration dataset needed). This is the easiest quantization method and works well for transformer models on CPU.

In [ ]:
# Apply dynamic quantization
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},  # Quantize all Linear layers
    dtype=torch.qint8,
)

quantized_size = get_model_size_mb(quantized_model)
print("=== DYNAMIC QUANTIZATION ===")
print(f"Original size:   {original_size:.1f} MB (FP32)")
print(f"Quantized size:  {quantized_size:.1f} MB (INT8)")
print(f"Reduction:       {original_size / quantized_size:.1f}x")
print(f"Savings:         {original_size - quantized_size:.1f} MB")

In [ ]:
# Verify quantized model predictions
print("=== QUANTIZED PREDICTIONS ===")
for text in test_texts:
    result = predict_sentiment(text, quantized_model, tokenizer)
    print(f"  '{text}'")
    print(f"    -> {result['label']} ({result['confidence']:.4f})")

### Accuracy Comparison

Let's systematically compare the original and quantized models on a larger set of examples to measure any accuracy degradation.

In [ ]:
# Compare accuracy on a test set
test_set = [
    ("The acting was superb and the story was compelling.", "POSITIVE"),
    ("A masterpiece of modern cinema.", "POSITIVE"),
    ("I loved every minute of this film.", "POSITIVE"),
    ("Heartwarming and beautifully crafted.", "POSITIVE"),
    ("One of the best movies I've seen this year.", "POSITIVE"),
    ("Boring and predictable plot.", "NEGATIVE"),
    ("The worst movie I have ever watched.", "NEGATIVE"),
    ("Complete waste of two hours.", "NEGATIVE"),
    ("Poorly written dialogue and bad acting.", "NEGATIVE"),
    ("I fell asleep halfway through.", "NEGATIVE"),
    ("Decent but forgettable.", "POSITIVE"),
    ("Not great, not terrible.", "NEGATIVE"),
    ("A solid effort with some flaws.", "POSITIVE"),
    ("Visually stunning but emotionally empty.", "NEGATIVE"),
    ("The soundtrack alone makes it worth watching.", "POSITIVE"),
]


def evaluate_model(
    model: torch.nn.Module,
    test_data: list[tuple[str, str]],
    tokenizer: AutoTokenizer,
) -> dict[str, float]:
    """Evaluate model accuracy on a test set.

    Args:
        model: PyTorch model to evaluate.
        test_data: List of (text, expected_label) tuples.
        tokenizer: Tokenizer for the model.

    Returns:
        Dictionary with accuracy and number of correct/total predictions.
    """
    correct = 0
    for text, expected_label in test_data:
        result = predict_sentiment(text, model, tokenizer)
        if result['label'] == expected_label:
            correct += 1
    return {
        'accuracy': correct / len(test_data),
        'correct': correct,
        'total': len(test_data),
    }


# Evaluate both models
original_results = evaluate_model(model, test_set, tokenizer)
quantized_results = evaluate_model(quantized_model, test_set, tokenizer)

print("=== ACCURACY COMPARISON ===")
accuracy_df = pd.DataFrame({
    "Model": ["Original (FP32)", "Quantized (INT8)"],
    "Correct": [original_results['correct'], quantized_results['correct']],
    "Total": [original_results['total'], quantized_results['total']],
    "Accuracy": [f"{original_results['accuracy']:.2%}", f"{quantized_results['accuracy']:.2%}"],
})
print(accuracy_df.to_string(index=False))

## Part 5: Speed Benchmarking

Size reduction is useful, but the real payoff for deployment is speed. Let's measure inference latency for both the original and quantized models with proper warmup and statistical rigor.

In [ ]:
def benchmark_model(
    model: torch.nn.Module,
    tokenizer: AutoTokenizer,
    text: str,
    num_warmup: int = 5,
    num_runs: int = 20,
) -> dict[str, float]:
    """Benchmark model inference latency.

    Args:
        model: PyTorch model to benchmark.
        tokenizer: Tokenizer for the model.
        text: Input text for inference.
        num_warmup: Number of warmup iterations.
        num_runs: Number of timed iterations.

    Returns:
        Dictionary with mean_ms, min_ms, max_ms latency statistics.
    """
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)

    # Warmup
    for _ in range(num_warmup):
        with torch.no_grad():
            _ = model(**inputs)

    # Measure
    times: list[float] = []
    for _ in range(num_runs):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model(**inputs)
        times.append(time.perf_counter() - start)

    return {
        'mean_ms': np.mean(times) * 1000,
        'min_ms': np.min(times) * 1000,
        'max_ms': np.max(times) * 1000,
    }


bench_text = "This is a sample text for benchmarking model inference speed."

print("Benchmarking original model...")
original_bench = benchmark_model(model, tokenizer, bench_text)

print("Benchmarking quantized model...")
quantized_bench = benchmark_model(quantized_model, tokenizer, bench_text)

print("\n=== SPEED COMPARISON ===")
speed_df = pd.DataFrame({
    "Model": ["Original (FP32)", "Quantized (INT8)"],
    "Size (MB)": [f"{original_size:.1f}", f"{quantized_size:.1f}"],
    "Mean Latency (ms)": [f"{original_bench['mean_ms']:.2f}", f"{quantized_bench['mean_ms']:.2f}"],
    "Min Latency (ms)": [f"{original_bench['min_ms']:.2f}", f"{quantized_bench['min_ms']:.2f}"],
    "Speedup": ["1.0x (baseline)", f"{original_bench['mean_ms']/quantized_bench['mean_ms']:.2f}x"],
})
print(speed_df.to_string(index=False))

## Part 6: ONNX Export and Optimization

ONNX (Open Neural Network Exchange) is a vendor-neutral format for ML models. Exporting to ONNX and running with ONNX Runtime often provides significant speedups because ONNX Runtime applies graph-level optimizations (operator fusion, memory planning) that PyTorch doesn't perform by default.

In [ ]:
if OPTIMUM_AVAILABLE:
    from optimum.onnxruntime import ORTModelForSequenceClassification

    # Export model to ONNX using Optimum (handles all the complexity)
    ONNX_PATH = "onnx_model"

    print("Exporting model to ONNX format...")
    ort_model = ORTModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        export=True,
    )
    ort_model.save_pretrained(ONNX_PATH)
    print(f"ONNX model saved to: {ONNX_PATH}/")

    # Check file sizes
    onnx_files = [f for f in os.listdir(ONNX_PATH) if f.endswith('.onnx')]
    for fname in onnx_files:
        fpath = os.path.join(ONNX_PATH, fname)
        size_mb = os.path.getsize(fpath) / (1024 ** 2)
        print(f"  {fname}: {size_mb:.1f} MB")
elif ONNX_AVAILABLE:
    print("Optimum not available. Demonstrating manual ONNX export.")
    print("Install optimum for easier export: pip install optimum[onnxruntime]")

    # Manual export
    ONNX_PATH = "onnx_model"
    os.makedirs(ONNX_PATH, exist_ok=True)

    dummy_input = tokenizer("test", return_tensors="pt", padding="max_length", max_length=128, truncation=True)

    print("Exporting to ONNX...")
    torch.onnx.export(
        model,
        (dummy_input['input_ids'], dummy_input['attention_mask']),
        os.path.join(ONNX_PATH, "model.onnx"),
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch', 1: 'sequence'},
            'attention_mask': {0: 'batch', 1: 'sequence'},
            'logits': {0: 'batch'},
        },
        opset_version=14,
    )
    onnx_size = os.path.getsize(os.path.join(ONNX_PATH, "model.onnx")) / (1024 ** 2)
    print(f"ONNX model exported: {onnx_size:.1f} MB")
else:
    print("ONNX not available. Skipping export.")
    print("Install with: pip install onnx onnxruntime")

In [ ]:
# Benchmark ONNX Runtime inference
if OPTIMUM_AVAILABLE:
    # Load the ONNX model
    ort_model = ORTModelForSequenceClassification.from_pretrained(ONNX_PATH)

    # Create pipeline with ONNX model
    onnx_pipeline = pipeline(
        "sentiment-analysis",
        model=ort_model,
        tokenizer=tokenizer,
    )

    # Verify predictions
    print("=== ONNX PREDICTIONS ===")
    for text in test_texts:
        result = onnx_pipeline(text)[0]
        print(f"  '{text}' -> {result['label']} ({result['score']:.4f})")

    # Benchmark
    print("\nBenchmarking ONNX Runtime...")

    # Warmup
    for _ in range(5):
        _ = onnx_pipeline(bench_text)

    onnx_times: list[float] = []
    for _ in range(20):
        start = time.perf_counter()
        _ = onnx_pipeline(bench_text)
        onnx_times.append(time.perf_counter() - start)

    onnx_mean_ms = np.mean(onnx_times) * 1000

    print(f"\nONNX Runtime mean latency: {onnx_mean_ms:.2f} ms")
    print(f"PyTorch mean latency:      {original_bench['mean_ms']:.2f} ms")
    print(f"Speedup:                   {original_bench['mean_ms']/onnx_mean_ms:.2f}x")
elif ONNX_AVAILABLE:
    # Manual ONNX Runtime benchmark
    session = ort.InferenceSession(os.path.join(ONNX_PATH, "model.onnx"))

    dummy = tokenizer(bench_text, return_tensors="np", padding="max_length", max_length=128, truncation=True)

    # Warmup
    for _ in range(5):
        _ = session.run(None, {"input_ids": dummy['input_ids'], "attention_mask": dummy['attention_mask']})

    onnx_times = []
    for _ in range(20):
        start = time.perf_counter()
        _ = session.run(None, {"input_ids": dummy['input_ids'], "attention_mask": dummy['attention_mask']})
        onnx_times.append(time.perf_counter() - start)

    onnx_mean_ms = np.mean(onnx_times) * 1000
    print(f"=== ONNX RUNTIME BENCHMARK ===")
    print(f"ONNX Runtime mean latency: {onnx_mean_ms:.2f} ms")
    print(f"PyTorch mean latency:      {original_bench['mean_ms']:.2f} ms")
    print(f"Speedup:                   {original_bench['mean_ms']/onnx_mean_ms:.2f}x")
else:
    onnx_mean_ms = None
    print("Skipping ONNX benchmark (not available).")

## Part 7: Complete Comparison

Let's bring all results together in a single comparison table. This gives a clear picture of the trade-offs between each compression technique and helps you decide which approach is best for your deployment scenario.

In [ ]:
# Final comparison table
results = [
    {
        "Model": "Original (FP32)",
        "Size (MB)": f"{original_size:.1f}",
        "Latency (ms)": f"{original_bench['mean_ms']:.2f}",
        "Size Reduction": "1.0x (baseline)",
        "Speedup": "1.0x (baseline)",
        "Accuracy": f"{original_results['accuracy']:.2%}",
    },
    {
        "Model": "Quantized (INT8)",
        "Size (MB)": f"{quantized_size:.1f}",
        "Latency (ms)": f"{quantized_bench['mean_ms']:.2f}",
        "Size Reduction": f"{original_size / quantized_size:.1f}x",
        "Speedup": f"{original_bench['mean_ms'] / quantized_bench['mean_ms']:.2f}x",
        "Accuracy": f"{quantized_results['accuracy']:.2%}",
    },
]

if onnx_mean_ms is not None:
    results.append({
        "Model": "ONNX Runtime",
        "Size (MB)": "~" + f"{original_size:.1f}",
        "Latency (ms)": f"{onnx_mean_ms:.2f}",
        "Size Reduction": "~1.0x",
        "Speedup": f"{original_bench['mean_ms'] / onnx_mean_ms:.2f}x",
        "Accuracy": f"{original_results['accuracy']:.2%}",
    })

print("=== COMPLETE COMPARISON ===")
comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

### When to Use Each Technique

Choosing the right compression technique depends on your deployment target, acceptable accuracy loss, and available tooling. Here's a decision guide.

In [ ]:
decision_guide = pd.DataFrame({
    "Scenario": [
        "Quick size reduction, GPU deployment",
        "CPU deployment, easy setup",
        "Maximum CPU speed",
        "Cross-platform deployment",
        "Mobile/edge devices",
    ],
    "Recommended Technique": [
        "FP16 conversion",
        "PyTorch dynamic quantization",
        "ONNX + quantization",
        "ONNX export",
        "INT4 quantization (GPTQ/AWQ)",
    ],
    "Complexity": ["Low", "Low", "Medium", "Medium", "High"],
    "Accuracy Impact": ["None", "Minimal (<1%)", "Minimal (<1%)", "None", "Small (1-3%)"],
})

print("=== DECISION GUIDE ===")
print(decision_guide.to_string(index=False))

### Cleanup

Let's clean up the ONNX model files we created during this notebook.

In [ ]:
# Clean up exported ONNX files
import shutil

if os.path.exists("onnx_model"):
    shutil.rmtree("onnx_model")
    print("Cleaned up onnx_model/ directory.")
else:
    print("No ONNX files to clean up.")

## Exercises

1. **Quantize a Larger Model**: Apply dynamic quantization to `bert-base-uncased` instead of DistilBERT. Compare the size reduction and speedup ratios.

2. **Batch Benchmarking**: Measure how quantization speedup changes with batch size (1, 8, 32, 64). Plot the results.

3. **Text Length Impact**: Benchmark both models with short (10 words), medium (50 words), and long (200 words) inputs. Does quantization help more with longer sequences?

4. **Save and Load Quantized**: Save the quantized model with `torch.save()` and reload it. Verify the file size on disk matches expectations.

5. **ONNX for Vision**: Export an image classification model (ViT) to ONNX and benchmark against the PyTorch version.

In [ ]:
# Try the exercises above to deepen your understanding of model compression.
# For example, quantize BERT-base:
#   bert_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased')
#   bert_quantized = torch.quantization.quantize_dynamic(
#       bert_model, {torch.nn.Linear}, dtype=torch.qint8
#   )
#   print(f"BERT size: {get_model_size_mb(bert_model):.1f} MB")
#   print(f"BERT quantized: {get_model_size_mb(bert_quantized):.1f} MB")

## Key Takeaways

- **FP16** halves model size with virtually no accuracy loss (best for GPU)
- **Dynamic quantization** reduces size 2-4x and speeds up CPU inference 1.5-3x
- **ONNX export** enables graph-level optimizations for 1.5-2x speedup
- **Accuracy degradation** from quantization is typically <1% for classification tasks
- **Measure before deploying** — always benchmark on your specific hardware and data

## Next Steps

- Try **Notebook 16**: MCP Basics (Agentic Workflows)
- Explore [HuggingFace Optimum](https://huggingface.co/docs/optimum/) for production optimization
- Learn about [bitsandbytes](https://github.com/TimDettmers/bitsandbytes) for 4-bit quantization (GPU)

## Resources

- [PyTorch Quantization](https://pytorch.org/docs/stable/quantization.html)
- [ONNX Runtime](https://onnxruntime.ai/)
- [HuggingFace Optimum](https://huggingface.co/docs/optimum/)
- [Quantization Overview (HuggingFace)](https://huggingface.co/docs/transformers/quantization)